# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [5]:
# Write your code below.
from dotenv import load_dotenv
import os
import shutil

# Now loading our. env file.
load_dotenv()

# Creating a variable that stores PRICE_DATA environment.
price_data = os.getenv("PRICE_DATA")

# We need to double check if PRICE_DATA has data.
if os.path.exists(price_data):
    print("PRICE_DATA loaded successfully.")
    print(price_data)
else:
    print("PRICE_DATA not found. Ensure you have executed the prerequisite notebook.")


PRICE_DATA loaded successfully.
../../05_src/data/prices/


In [6]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [7]:
import os
from glob import glob

# Write your code below.

parquet_files = glob(os.path.join(price_data, "**/*.parquet"), recursive = True)
#print(parquet_files)

# Print out the first parquet file's dataset.
ddf = dd.read_parquet(parquet_files[0])
ddf.compute()


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
CTAS,2002-01-02,NaN,5.070603,5.109207,4.913060,5.055997,3887200.0,2002
CTAS,2002-01-03,NaN,5.143637,5.148854,5.005917,5.110251,2285600.0,2002
CTAS,2002-01-04,NaN,5.145725,5.173895,5.011135,5.135292,1502400.0,2002
CTAS,2002-01-07,NaN,5.106078,5.216672,5.100862,5.136335,2756800.0,2002
CTAS,2002-01-08,NaN,5.211456,5.232323,5.117556,5.138422,2806400.0,2002
...,...,...,...,...,...,...,...,...
CTAS,2002-12-24,NaN,4.978729,5.002168,4.954224,4.995776,1637200.0,2002
CTAS,2002-12-26,NaN,4.974468,5.133216,4.956356,4.976599,2782000.0,2002
CTAS,2002-12-27,NaN,4.862598,4.994711,4.839159,4.960618,3041600.0,2002


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [14]:
# Write your code below.

# Reading the first 10 parquet files.
# Reason: It would take a very long time to get all the data, so I am using the first 10 files.
ddf = dd.read_parquet(parquet_files[0:10]).set_index("Ticker")

# Now we need to add the 'lag' feature for Close and Adj_Close according to the question description.
ddf["Close_lag_1"] = ddf.groupby("Ticker")["Close"].shift(1)
ddf["Adj_Close_lag_1"] = ddf.groupby("Ticker")["Adj Close"].shift(1)

# Now we get 'returns'.
ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1

# Now we calculate the range by using the high subtracting from the low.
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

dd_feat = ddf

dd_feat.compute()


/var/folders/t9/j9cv6kkj7pg15qwk9ljc1ttr0000gn/T/ipykernel_80606/1964006483.py:8: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Close_lag_1"] = ddf.groupby("Ticker")["Close"].shift(1)
/var/folders/t9/j9cv6kkj7pg15qwk9ljc1ttr0000gn/T/ipykernel_80606/1964006483.py:9: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Adj_Close_lag_1"] = ddf.groupby("Ticker")["Adj Close"].shift(1)


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,,
CTAS,2019-01-02,NaN,37.263641,37.283664,36.618439,36.823123,2378400.0,2019,NaN,NaN,NaN,2.823963
CTAS,2019-01-03,NaN,36.667389,37.822075,36.598420,37.363761,3674000.0,2019,77.999481,NaN,0.015396,2.410810
CTAS,2019-01-04,NaN,38.391632,38.438355,36.863176,37.199124,3482000.0,2019,79.200363,NaN,0.006019,1.670773
CTAS,2019-01-07,NaN,38.456158,38.749833,37.980045,38.376064,2956400.0,2019,79.677071,NaN,0.011596,1.865995
CTAS,2019-01-08,NaN,38.872189,39.045725,38.075700,38.829916,3110000.0,2019,80.600967,NaN,-0.013828,2.165646
...,...,...,...,...,...,...,...,...,...,...,...,...
CTAS,2004-12-27,NaN,4.760652,4.836567,4.711142,4.813462,2999600.0,2004,4.811263,NaN,0.034807,0.047944
CTAS,2004-12-28,NaN,4.803559,4.837666,4.726544,4.761751,2940800.0,2004,4.760652,NaN,0.044913,0.176860
CTAS,2004-12-29,NaN,4.838769,4.882778,4.794760,4.825566,3293200.0,2004,4.803559,NaN,0.012291,0.155552


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

dd_feat["returns_10_days"] = dd_feat.groupby("Ticker").apply(lambda x: x["returns"].rolling(10).mean())

#ddf.head()


/var/folders/t9/j9cv6kkj7pg15qwk9ljc1ttr0000gn/T/ipykernel_80606/91800989.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["returns_10_days"] = ddf.groupby("Ticker").apply(lambda x: x["returns"].rolling(10).mean())


TypeError: Column assignment doesn't support type <class 'dask_expr._collection.DataFrame'>

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

Kevin's Answer: No, converting the Dask DataFrame to a Pandas DataFrame was not strictly required to compute the moving average. Since Dask supports rolling operations, we could have directly used .rolling(10).mean() within Dask.

+ Would it have been better to do it in Dask? Why?

Kevin's Answer: Yes, by computing the moving average in Dask, it would have been a better approach, particularly for large datasets. Dask offers greater scalability by handling data that exceeds memory limits, whereas Pandas requires loading everything into RAM. Additionally, Dask leverages parallelism by distributing computations across multiple cores or even a cluster, while Pandas operates in a single-threaded manner. This makes Dask more efficient, as converting a large dataset to Pandas can be both slow and memory-intensive.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.